# 🤝 PE Deal Intelligence

*Public private-equity deal announcements, turned into an evidence-backed deal record for a portfolio operations team.*

**User.** A portfolio operations manager at a lower-middle-market sponsor that buys and builds industrial, distribution and
business-services companies. **Recurring task.** Before each add-on planning cycle: which add-ons are peer sponsors'
platforms buying, what capabilities do they say those add-ons bring, and how fast do the best builders close deals?

This notebook runs the whole workflow:

1. **Setup** — fresh Colab session: clone the project, install dependencies
2. **Collect** — the four firms' news pages (polite: robots.txt and crawl delays), or the saved pages (default)
3. **Process** — parse, tell deals from other news, extract every field with its evidence, handle duplicates and gaps
4. **Inspect** — the data dictionary, a record next to its source sentences, coverage and failed pages
5. **Validate** — accuracy against hand-labelled pages (held-out vs development), consistency checks, repeatability
6. **Decide** — the add-on research brief, the buy-and-build tracker, charts
7. **Extras** — similar-deal search, the new-deal digest, exports

> **Cached data.** By default everything runs on pages saved by the collector (`data/pe_deals/cache/`, collection time
> in every record), so the notebook gives the same result even if a website changes or goes offline. Set `LIVE = True`
> in step 2 to fetch new pages first (slow by design: about 10 seconds between pages on two of the sites).

The same product is the **🤝 PE deals** tab of the web app (`streamlit run app.py`). Method, data dictionary and
limitations: `docs/PE_DEALS.md`.


## 1 · Setup

In [1]:
# On Google Colab this clones the project; inside the repo (or its notebooks/ folder) it just finds the package.
import os, sys, subprocess, pathlib
REPO_URL = "https://github.com/lotainc0/Stock-Trader-FINTECH-Project.git"
here = pathlib.Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "pedeals" / "__init__.py").exists()), None)
if root is None:
    subprocess.run(["git", "clone", "-q", REPO_URL, "lighthouse_repo"], check=True)
    root = here / "lighthouse_repo"
os.chdir(root); sys.path.insert(0, str(root))
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "requests", "beautifulsoup4", "lxml",
                    "plotly"], check=True, timeout=900)
except Exception as exc:  # offline or already installed - keep going
    print("pip install skipped:", exc)
import pandas as pd
from IPython.display import Markdown, display
import pedeals
from pedeals import build_dataset, analytics as an, charts
from pedeals.sources import SOURCES
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
print("PE Deal Intelligence", pedeals.__version__, "ready in", root)

PE Deal Intelligence 1.0.0 ready in /home/user/Stock-Trader-FINTECH-Project


## 2 · Collect

The four firms are peers of the user (lower-middle-market, Midwest / central US, explicit buy-and-build) and each lists
its news in an XML sitemap. The collector reads each site's robots.txt and obeys it, waits the site's crawl delay
between requests, names itself in the User-Agent, saves every page with its fetch time and logs every URL's outcome.

In [2]:
pd.DataFrame([{"key": k, "firm": s.firm, "home office": s.hq, "sitemaps": ", ".join(s.sitemaps)} for k, s in SOURCES.items()])

,key,firm,home office,sitemaps
0,tonkabay,Tonka Bay Equity Partners,"Minnetonka, MN",https://tonkabayequity.com/wp-sitemap-posts-news-1.xml
1,goldnerhawn,Goldner Hawn,"Minneapolis, MN",https://goldnerhawn.com/wp-sitemap-posts-news-1.xml
2,huron,Huron Capital,"Detroit, MI",https://www.huroncapital.com/news-sitemap.xml
3,align,Align Capital Partners,"Dallas, TX and Cleveland, OH",https://aligncp.com/post-sitemap.xml


In [3]:
LIVE = False           # True: fetch new articles from the live sites first (minutes; pages already saved are skipped)
START = "2023-01-01"   # first publication date of interest

if LIVE:
    from pedeals.collect import collect
    discovery = collect(start=START)
    display(discovery.groupby(["source", "outcome"]).size().rename("urls").to_frame())
else:
    print("Using the saved pages (CACHED DATA). Set LIVE = True to refresh.")

Using the saved pages (CACHED DATA). Set LIVE = True to refresh.


## 3 · Process

`build_dataset` parses every saved page, finds its publication date (page metadata → dateline → the firm's news-page
listing → first date in the text; never guessed), classifies it (deal, personnel, fund / firm, recognition, other),
extracts the deal fields from the release's lead sentence with the evidence for each, infers ownership from the
firm's other announcements where the release is silent, marks duplicates and keeps the period.

In [4]:
ds = build_dataset(start=START)
print(ds.summary_text())

PE Deal Intelligence dataset (CACHED DATA: built from pages saved by `python -m pedeals collect` (fetch time in `collected_at`); the live sites may have changed since.)
period: published 2023-01-01 to 2026-10-13; built 2026-10-29T15:41:58+00:00
185 articles parsed from 4 firms; 115 deal announcements in the period, 115 unique deals, 0 failed pages
publisher                  transaction_type            
Align Capital Partners     Add-on acquisition              51
                           Exit                             5
                           Growth / minority investment     2
                           Merger / combination             2
                           Platform investment             10
Goldner Hawn               Add-on acquisition              11
                           Exit                             3
                           Platform investment              3
Huron Capital              Add-on acquisition              10
                           Exit     

In [5]:
# Coverage: what each sitemap listed, what was skipped and why, what was kept
ds.coverage

,source,firm,hq,sitemap_urls,skipped_before_period,skipped_robots,pages_parsed,failed_pages,outside_period,articles_in_period,deals_in_period,non_deal_news_in_period,duplicates_marked,earliest_in_period,latest_in_period,collected_at
0,tonkabay,Tonka Bay Equity Partners,"Minnetonka, MN",131,82,0,49,0,23,26,15,11,0,2023-01-04,2026-07-07,2026-10-29T15:08:33+00:00
1,goldnerhawn,Goldner Hawn,"Minneapolis, MN",21,0,0,21,0,0,21,17,4,0,2024-01-05,2026-10-13,2026-10-29T15:03:53+00:00
2,huron,Huron Capital,"Detroit, MI",195,167,0,28,0,11,17,13,4,0,2023-06-08,2026-07-01,2026-10-29T15:02:38+00:00
3,align,Align Capital Partners,"Dallas, TX and Cleveland, OH",171,84,0,87,0,3,84,70,14,0,2023-01-05,2026-10-07,2026-10-29T15:07:34+00:00


In [6]:
# Pages that could not be fetched or parsed (with the reason), and articles with no publication date
display(ds.failures)
ds.records.loc[ds.records["published_date"] == "", ["publisher", "headline", "period_note"]]

,source,url,stage,reason,fetched_at


,publisher,headline,period_note
0,Tonka Bay Equity Partners,Corporate Technologies Acquires CCSI,no publication date found
1,Tonka Bay Equity Partners,Fenix Specialty Coatings and National Coatings Inc. Join Forces,no publication date found
2,Tonka Bay Equity Partners,Vince Thao Named a “Rising Star” by ACG MN,no publication date found
3,Huron Capital,2021 Year in Review,no publication date found (last modified 2023-01-18)
4,Huron Capital,2022 Year-In-Review: Huron Capital’s Momentum Continues,no publication date found (last modified 2023-01-18)
5,Huron Capital,2023 Year-In-Review: Huron Capital Wraps Up Record Year of ExecFactor® Activity,no publication date found (last modified 2024-05-31)
6,Huron Capital,Huron Capital & HCP Services 2024 Year in Review,no publication date found (last modified 2025-02-19)
7,Huron Capital,Huron Capital Invests in RK Electric,no publication date found (last modified 2024-03-13)
8,Huron Capital,Huron Capital Launches Rampart Exterior Services to Provide Commercial Roofi...,no publication date found (last modified 2024-07-30)
9,Huron Capital,Huron Capital Partners Celebrates 25th Anniversary of Building Premier Compa...,no publication date found (last modified 2024-06-26)


## 4 · Inspect

### Data dictionary
Every output field is defined in `docs/PE_DEALS.md` §5. The categories are defined in code, in one place:

In [7]:
from pedeals.taxonomy import NEWS_CATEGORIES, TRANSACTION_TYPES, DEAL_STATUS, INDUSTRIES, RATIONALE_TAGS
display(pd.DataFrame(TRANSACTION_TYPES.items(), columns=["transaction_type", "definition"]))
display(pd.DataFrame(DEAL_STATUS.items(), columns=["deal_status", "definition"]))
display(pd.DataFrame(NEWS_CATEGORIES.items(), columns=["news_category", "definition"]))
pd.DataFrame([(k, ", ".join(v[:8]) + " ...") for k, v in INDUSTRIES.items()], columns=["target_industry", "example stems"])

,transaction_type,definition
0,Platform investment,the publishing firm itself acquires or invests in a company (a new platform ...
1,Add-on acquisition,a company the firm already owns (a portfolio / platform / partner company) a...
2,Growth / minority investment,"the firm invests without taking control (minority, growth equity, structured..."
3,Merger / combination,"two companies join forces or merge, at least one backed by the firm"
4,Exit,"the firm (or a fund it manages) sells a portfolio company, or a portfolio co..."
5,Dividend recapitalization,a portfolio company refinances to return capital to its owners (no new owner)
6,Unclear,the text describes a transaction but its type cannot be read reliably


,deal_status,definition
0,Completed,"the text says the transaction has happened (has acquired, completed, closed,..."
1,Announced / pending,"signed or agreed but not stated as closed (definitive agreement, will acquir..."
2,Unknown,neither is stated


,news_category,definition
0,Deal,"a transaction: an investment, acquisition, sale, recapitalization or combina..."
1,Personnel,"hires, promotions, appointments, board changes"
2,Fund / firm,"fund closes, new offices, firm milestones, anniversaries"
3,Recognition / media,"awards, rankings, podcasts, interviews, articles, events"
4,Other,"anything else (hiring notices, thought leadership, community news)"


,target_industry,example stems
0,Industrial manufacturing,"manufactur, fabricat, machin, precision, component, bearing, coating, moldin..."
1,Value-added distribution,"distribut, wholesale, supply house, fulfillment, kitting ..."
2,Aerospace & defense,"aerospace, defense, aircraft, aviation, military, space ..."
3,IT services & cybersecurity,"managed service, managed it, it service, it solution, msp, cybersecurity, cy..."
4,Software & data,"software, saas, analytics, data platform, digital platform, application, gis..."
5,Business services,"staffing, consult, outsourc, collection, receivable, screening, background c..."
6,Healthcare,"healthcare, health care, medical, dental, dermatolog, clinic, physician, pat..."
7,Building & infrastructure services,"construction, hvac, plumbing, roofing, electrical contract, electrical servi..."
8,Transportation & logistics,"logistics, trucking, freight, transportation, fleet, warehous, towing, deliv..."
9,Consumer & food,"consumer, retail, restaurant, food, beverage, apparel, produce, pet ..."


### The brief's example, as a record

Tonka Bay published the Standard Locknut / Jones Bearing announcement; Tonka Bay is **not** the buyer. Each value
below sits next to the words it was read from.

In [8]:
deals = ds.unique_deals
jones = deals[deals["headline"].str.contains("Jones Bearing")].iloc[0]
Markdown(an.deal_card(jones))

#### Standard Locknut acquires Jones Bearing

| Field | Value | Evidence |
|---|---|---|
| Publisher (PE firm) | Tonka Bay Equity Partners | source page: https://tonkabayequity.com/news/standard-locknut-acquires-jones-bearing/ |
| Published | 2026-07-07 | dateline: MINNETONKA, MN (July 7, 2026) — |
| Transaction type | Add-on acquisition | the buyer is a Tonka Bay Equity Partners company: MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Buyer / investor | Standard Locknut | MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Buyer's relationship to the publisher | Tonka Bay Equity Partners portfolio company | the buyer is a Tonka Bay Equity Partners company: MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Target | Jones Bearing | MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Target's business | Based in Alabama, Jones Bearing is a manufacturer of mounted roller bearings and power transmission components used in industrial applications. | announcement text |
| Target industry (inferred) | Industrial manufacturing | stems matched: manufactur, component, bearing |
| Target's customers / end markets | Industrial / OEM | from 'serving', 'used in', 'customers in' phrases |
| Target location | Alabama (AL) | — |
| Stated rationale | The acquisition expands Standard Locknut’s product offering into light-to-medium-duty mounted bearings while broadening its reach across both OEM and distributor channels. | tags: Product / service expansion; Customers / channels |
| Deal status | Completed | Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Closing date | Not stated | the publication date is not a closing date |
| Price | Not stated | — |
| Ownership | Unknown | — |

*Review flags:* date from dateline

In [9]:
# All deal announcements in the period (the main columns)
deals[an.DISPLAY_COLUMNS].head(15)

,published_date,publisher,transaction_type,buyer,target,target_industry,target_state,rationale_tags,price_disclosure,deal_status,url
0,2023-01-04,Tonka Bay Equity Partners,Exit,Unknown,Engineered Specialty Products,Industrial manufacturing,GA,Product / service expansion; Technology / capability,Not stated,Unknown,https://tonkabayequity.com/news/tonka-bay-announces-sale-of-esp/
1,2023-01-05,Align Capital Partners,Platform investment,Align Capital Partners,Cleartelligence,Software & data,,Product / service expansion; Customers / channels; Technology / capability; ...,Not stated,Unknown,https://aligncp.com/news/acp-believes-in-the-power-of-data-with-growth-inves...
2,2023-01-12,Align Capital Partners,Exit,Champlain Capital,iPROMOTEu,Business services,MA,Technology / capability; Capacity / scale; Talent / leadership,Not stated,Completed,https://aligncp.com/news/align-capital-partners-completes-sale-of-ipromoteu/
3,2023-01-19,Align Capital Partners,Add-on acquisition,StenTech,"Advanced Tooling Design, Inc",Industrial manufacturing,CA,Geographic reach; Customers / channels; Capacity / scale,Not stated,Completed,https://aligncp.com/news/stentech-expands-into-silicon-valley-region-with-ac...
4,2023-02-02,Align Capital Partners,Add-on acquisition,"Schneider Geospatial, LLC",WebGUIDE Xtreme,Software & data,WI,Geographic reach; Customers / channels; Talent / leadership,Not stated,Completed,https://aligncp.com/news/acp-govtech-platform-schneider-geospatial-acquires-...
5,2023-02-28,Align Capital Partners,Add-on acquisition,Marco Rubber & Plastics,Jet Gasket & Seal,Value-added distribution,NV,Product / service expansion; Customers / channels,Not stated,Completed,https://aligncp.com/news/marco-expands-into-brewery-end-market-with-add-on-o...
6,2023-06-07,Align Capital Partners,Add-on acquisition,"Schneider Geospatial, LLC",PeopleGIS,Software & data,MA,Technology / capability; Talent / leadership,Not stated,Completed,https://aligncp.com/news/schneider-geospatial-strengthens-govtech-leadership...
7,2023-06-08,Huron Capital,Add-on acquisition,Horsepower Automotive,Flatline Van Company,Consumer & food,,Product / service expansion; Geographic reach; Technology / capability,Not stated,Unknown,https://www.huroncapital.com/news/horsepower-automotive-partners-with-flatli...
8,2023-07-18,Tonka Bay Equity Partners,Platform investment,Tonka Bay Equity Partners,Corporate Technologies,IT services & cybersecurity,MN,Geographic reach; Customers / channels; Technology / capability,Not stated,Unknown,https://tonkabayequity.com/news/tonka-bay-invests-in-corporate-technologies/
9,2023-08-09,Align Capital Partners,Add-on acquisition,"Schneider Geospatial, LLC",Spatialest,Software & data,,Product / service expansion; Technology / capability,Not stated,Completed,https://aligncp.com/news/schneider-geospatial-continues-growth-journey-with-...


## 5 · Validate

109 articles were labelled field by field from the source pages. The rules were improved on 71 (**development**,
in-sample) and then frozen in git before 38 more were drawn and labelled (**held-out**). Only the held-out accuracy is
the tool's accuracy. Consistency checks run on every record; repeatability compares two builds.

In [10]:
from pedeals.validate import full_report
Markdown(full_report(ds))

# PE Deal Intelligence: validation

Reference labels: 38 held-out articles (labelled after the extraction rules were frozen in git) and 71 development articles (the rules were tuned on these, so their accuracy is in-sample).

| Field | Held-out: correct / labelled | Held-out accuracy (95% interval) | Development accuracy |
|---|---|---|---|
| is_deal | 38 / 38 | 100% (91%-100%) | 100% (71/71) |
| news_category | 38 / 38 | 100% (91%-100%) | 100% (71/71) |
| published_date | 38 / 38 | 100% (91%-100%) | 100% (71/71) |
| transaction_type | 30 / 30 | 100% (89%-100%) | 100% (54/54) |
| buyer | 30 / 30 | 100% (89%-100%) | 100% (54/54) |
| target | 30 / 30 | 100% (89%-100%) | 98% (53/54) |
| target_industry | 26 / 30 | 87% (70%-95%) | 92% (49/53) |
| target_state | 29 / 30 | 97% (83%-99%) | 96% (52/54) |
| deal_status | 29 / 30 | 97% (83%-99%) | 100% (54/54) |
| price_disclosure | 30 / 30 | 100% (89%-100%) | 100% (54/54) |

Deal vs other news (held-out): precision 100%, recall 100% (30 deals found, 0 false alarms, 0 missed).

## Consistency checks (every record, no labels needed)

| Check | Records failing |
|---|---|
| publication date after the collection date | 0 |
| publication date in the future | 0 |
| deal without a source URL | 0 |
| deal without category evidence | 0 |
| publisher recorded as an add-on buyer | 0 |
| platform investment not by the publisher | 0 |
| buyer and target the same | 0 |
| price recorded as zero | 0 |
| price disclosed without a value | 0 |
| closing date before the publication date by > 1 year | 0 |
| duplicate deal ids | 0 |
| deal with a type but no buyer and no target | 0 |

Repeatability: two builds from the same saved pages are identical (fingerprint `7292faa4721f990f`).

## Mismatches, held-out (6)

| Article | Field | Label | Program |
|---|---|---|---|
| Align Partners with Engineering Services Firm AKS | target_state | OR |  |
| Counsel Press Expands into the Trial Courts with Case Anywhere Acquisi | target_industry | Software & data | IT services & cybersecurity |
| Marco Sealing Solutions Continues Expansion with Acquisition of Pride  | target_industry | Value-added distribution | Industrial manufacturing |
| Schneider Geospatial Expands Florida Presence | target_industry | Software & data | Unknown |
| Schneider Geospatial Expands Florida Presence | deal_status | Unknown | Completed |
| Huron Capital’s Platform 1-800 HANSONS Acquires Paramount Builders, LL | target_industry | Building & infrastructure services | Consumer & food |

## Mismatches, development (7)

| Article | Field | Label | Program |
|---|---|---|---|
| Align Capital Partners Announces Sale of CompletePet to Blue Sea Capit | target_industry | Consumer & food | Industrial manufacturing |
| Counsel Press Rebrands as Proceed, Broadens Market Reach with Three Ac | target_industry | Business services | Transportation & logistics |
| Counsel Press Rebrands as Proceed, Broadens Market Reach with Three Ac | target_state | none | TX |
| Premier Biotech Increases Offerings and Global Distribution Network wi | target | OraSure Toxicology Line; Green Earth Biomedical | OraSure Technologies, Inc |
| Goldner Hawn Forms Nexterra Environmental, A New Erosion Control and S | target_industry | Value-added distribution | Energy & environmental |
| Professional Acquires “Project Heat” | target_industry | Business services | Unknown |
| Huron Capital’s ExecFactor® Platform The Exigent Group Acquires Smith- | target_state | OH |  |

## Notes

- Every labelled URL is in the dataset.
- Labels were made by reading each source page; see data/pe_deals/validation/README.md for who labelled what and how. The intervals are 95% Wilson intervals: with 30-50 labelled deals they are wide.


## 6 · Decide: the add-on research brief

*"Our platform is an IT managed-services provider. Which capabilities are peer platforms adding, and where should our
next add-on search start?"* Change `INDUSTRY` to your platform's industry.

In [11]:
INDUSTRY = "IT services & cybersecurity"
Markdown(an.research_brief(deals, INDUSTRY, "Which capabilities are peer platforms adding, and which should we investigate?"))

### Add-on research brief: IT services & cybersecurity

*Question:* Which capabilities are peer platforms adding, and which should we investigate?

15 add-on acquisitions by 5 platforms of 3 sponsors (2023-09-20 to 2026-04-30).

**What the buyers said the add-ons add** (share of add-ons that state any rationale):

- Technology / capability: 12 (80%)
- Geographic reach: 8 (53%)
- Product / service expansion: 6 (40%)
- Customers / channels: 5 (33%)
- Capacity / scale: 3 (20%)
- Talent / leadership: 2 (13%)

**The deals, with the evidence:**

- 2026-04-30 · **Corporate Technologies** (Tonka Bay Equity Partners) acquired **RPM Technologies, LLC** (IN): RPM is Corporate Technologies’ 5th acquisition since partnering with Tonka Bay and expands its footprint to 21 locations across the U.S. [source](https://tonkabayequity.com/news/corporate-technologies-acquires-rpm/)
- 2026-03-10 · **ISPN** (Align Capital Partners) acquired **ZCorum** (GA): Align Capital Partners’ (“ACP”) portfolio company ISPN (or the “Company”) completed the acquisition of ZCorum, positioning the combined companies as a market leader in broadband-focused managed IT services. Headquartered in Alpharetta, Georgia, ZCorum … [source](https://aligncp.com/news/ispn-and-zcorum-unite-to-expand-managed-services-for-broadband-providers/)
- 2025-10-29 · **Cyber Advisors** (Goldner Hawn) acquired **Infranet Technologies Group**: Founded in 1998, Infranet Technologies Group has built a strong reputation for delivering high-quality IT consulting services, specializing in enterprise infrastructure, cloud solutions, and managed services. The acquisition of Infranet Technologies Group … [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-completes-tenth-acquisition-acquiring-infranet-technologies-group/)
- 2025-09-03 · **Corporate Technologies** (Tonka Bay Equity Partners) acquired **Cenetric**: Cenetric is Corporate Technologies’ 3 rd acquisition since partnering with Tonka Bay in 2023 and expands its footprint to 18 locations across the U.S. [source](https://tonkabayequity.com/news/corporate-technologies-acquires-cenetric/)
- 2025-04-22 · **Cyber Advisors** (Goldner Hawn) acquired **Stratum Security** (VA): This strategic move highlights Cyber Advisors’ dedication to expanding its cybersecurity capabilities and delivering comprehensive offensive security services to clients nationwide. The integration of Stratum Security’s seasoned team into Cyber Advisors will … [source](https://goldnerhawn.com/news/cyber-advisors-acquires-stratum-security/)
- 2025-02-27 · **Cyber Advisors** (Goldner Hawn) acquired **Wasatch I.T** (UT): This strategic acquisition marks a significant milestone in Cyber Advisors’ expansion nationwide, this partnership extends its reach to the western United States. The acquisition of Wasatch I.T. represents a significant milestone for Cyber Advisors, marking … [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-acquires-managed-service-provider-wasatch-i-t/)
- 2024-12-20 · **Counsel Press Inc** (Align Capital Partners) acquired **Case Anywhere LLC**: Align Capital Partners’ (“ACP”) outsourced legal services platform Counsel Press Inc. (“Counsel Press” or the “Company”) adds trial court services with its acquisition of Case Anywhere LLC (“Case Anywhere”). The addition of Case Anywhere strengthens Counsel … [source](https://aligncp.com/news/counsel-press-expands-into-the-trial-courts-with-case-anywhere-acquisition/)
- 2024-11-06 · **Cyber Advisors** (Goldner Hawn) acquired **eDot** (IL): This strategic acquisition bolsters Cyber Advisors’ plan to broaden its service reach and technical expertise in the Midwest, offering industry-leading Managed Service Solutions complemented by a comprehensive Cyber Security offering. The acquisition of eDot … [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-acquires-managed-service-provider-edot/)
- 2024-10-16 · **Corporate Technologies** (Tonka Bay Equity Partners) acquired **RSPN** (IA): RSPN is Corporate Technologies’ second acquisition since partnering with Tonka Bay in 2023 and expands its footprint to 17 cities across the U.S. [source](https://tonkabayequity.com/news/corporate-technologies-acquires-rspn/)
- 2024-08-21 · **Cyber Advisors** (Goldner Hawn) acquired **BAI Security** (IL): This strategic purchase solidifies Cyber Advisors’ commitment to market-leading cybersecurity capabilities and expands its reach in the ever-evolving landscape of digital security, risk, and compliance. Their combined expertise will enable clients to remain … [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-acquires-bai-security/)
- 2024-06-03 · **Cyber Advisors** (Goldner Hawn) acquired **Total Networx** (MN): The acquisition of Total Networx represents yet another milestone for Cyber Advisors, further reinforcing its position as a trusted advisor to organizations seeking comprehensive cybersecurity and managed IT solutions. With a reputation for delivering premium … [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-acquires-total-networx/)
- 2024-05-09 · **Cyber Advisors** (Goldner Hawn) acquired **S&L Computer Services** (ND): The addition of SLC to the Cyber Advisors’ family brings expanded scale and depth for the organization along with access to marquee customers. Together, their combined expertise will empower clients to thrive without the weight of technology challenges, … [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-completes-acquisition-of-sl-computer-services/)
- 2024-02-13 · **Cyber Advisors** (Goldner Hawn) acquired **Innovative Technology Partners** (MN): The acquisition of Innovative Technology Partners further reinforces Cyber Advisors’ position as a trusted advisor to organizations seeking comprehensive cybersecurity and managed IT solutions. [source](https://goldnerhawn.com/news/cyber-advisors-completes-acquisition-of-innovative-technology-partners/)
- 2024-01-30 · **Corporate Technologies** (Tonka Bay Equity Partners) acquired **NuMSP**: The acquisition expands Corporate Technologies’ national presence to 16 markets in eleven states. NuMSP clients will benefit from expanded service offerings including 24×7 NOC and SOC services and comprehensive cybersecurity solutions. [source](https://tonkabayequity.com/news/corporate-technologies-acquires-numsp/)
- 2023-09-20 · **E Source** (Align Capital Partners) acquired **Utilligent LLC**: Utilligent brings deep utility expertise and a data-driven approach that furthers E Source’s capabilities in areas such as smart infrastructure, utility operations, technology & the digital utility and customer engagement. The acquisition adds incremental … [source](https://aligncp.com/news/e-source-acquires-leading-utilities-advisor-utilligent/)

**Supported by this evidence:** which capabilities peer platforms chose to buy and how they described them. **Needs further research:** whether those add-ons improved margins, growth or returns (operating results, management interviews, lender data), deals the firms did not announce, and the prices paid (rarely disclosed).

In [12]:
# Buy-and-build tracker: add-ons per platform and how fast they come
bb = an.buy_and_build(deals)
bb.drop(columns=["platform_key"]).head(12)

,platform,sponsor,add_ons,companies_acquired,first,last,median_days_between,targets,target_industries,target_states,rationale
0,"Schneider Geospatial, LLC",Align Capital Partners,8,8,2023-02-02,2026-06-24,125.0,WebGUIDE Xtreme; PeopleGIS; Spatialest; Systems Development Group; Full Circ...,Software & data,CT; IL; MA; WI,Technology / capability (5); Geographic reach (3); Talent / leadership (3); ...
1,Cyber Advisors,Goldner Hawn,8,8,2024-02-13,2025-10-29,79.0,Innovative Technology Partners; S&L Computer Services; Total Networx; BAI Se...,IT services & cybersecurity,IL; MN; ND; UT; VA,Technology / capability (6); Product / service expansion (4); Customers / ch...
2,Marco Sealing Solutions / Marco Rubber & Plastics,Align Capital Partners,7,7,2023-02-28,2026-09-03,172.5,Jet Gasket & Seal; American Seal & Packing; Allied Metrics O-Rings & Seals; ...,Industrial manufacturing; Value-added distribution,CA; CO; NV; TX,Customers / channels (6); Product / service expansion (4); Technology / capa...
3,E Source,Align Capital Partners,6,6,2023-09-20,2026-08-24,105.0,Utilligent LLC; UMS Group; ILLUME Advising; Opinion Dynamics; CF Power; Aned...,Business services; Energy & environmental; IT services & cybersecurity,,Customers / channels (5); Technology / capability (5); Talent / leadership (...
4,Counsel Press Inc,Align Capital Partners,6,10,2024-09-19,2025-11-18,70.0,Cockle Legal Briefs; Becker Gallagher Briefs and Records; Lantagne Legal Pri...,Business services; IT services & cybersecurity; Transportation & logistics,IL; NY; TX; VA,Geographic reach (5); Customers / channels (3); Technology / capability (3);...
5,Premier Biotech,Align Capital Partners,5,6,2024-06-06,2026-06-18,182.0,"Desert Tox, LLC; OraSure Technologies, Inc; National Test Systems; GH Soluti...",Healthcare; Value-added distribution,,Customers / channels (5); Product / service expansion (4); Talent / leadersh...
6,MKB Company,Align Capital Partners,4,4,2025-05-21,2026-09-09,188.0,"Silt Saver, Inc; Enviroscape; US Silt & Site Supply; Siltworm",Industrial manufacturing,GA; IN; NH; OH,Product / service expansion (4); Geographic reach (3); Capacity / scale (2);...
7,Corporate Technologies,Tonka Bay Equity Partners,4,4,2024-01-30,2026-04-30,260.0,"NuMSP; RSPN; Cenetric; RPM Technologies, LLC",IT services & cybersecurity,IA; IN,Geographic reach (4); Technology / capability (4); Product / service expansi...
8,StenTech,Align Capital Partners,4,4,2023-01-19,2026-04-21,215.0,"Advanced Tooling Design, Inc; Photo Etch Technology; Advanced Metal Etching,...",Consumer & food; Industrial manufacturing,CA; OR,Geographic reach (3); Capacity / scale (3); Product / service expansion (2);...
9,The Exigent Group,Huron Capital,3,4,2023-10-31,2026-06-01,472.0,Ambient Temperature Corp; Global Facility Solutions; Premier Mechanical; Sup...,Building & infrastructure services,MA; OH,Geographic reach (3); Product / service expansion (1); Customers / channels ...


In [13]:
adds = deals[deals["transaction_type"] == "Add-on acquisition"].merge(bb[["platform_key", "platform"]], on="platform_key")
charts.buy_and_build_timeline(adds, bb[bb["add_ons"] >= 2]["platform"].tolist()).show()

In [14]:
charts.deals_by_quarter(deals).show()
rf = an.rationale_frequency(deals[deals["transaction_type"] == "Add-on acquisition"])
charts.horizontal_counts(rf, "tag", "deals", "Why add-ons were made (stated rationale)", "add-ons").show()
display(an.rationale_frequency(deals[deals["transaction_type"] == "Add-on acquisition"], by="target_industry"))

tag,Product / service expansion,Geographic reach,Customers / channels,Technology / capability,Capacity / scale,Talent / leadership
target_industry,,,,,,
Building & infrastructure services,3,7,4,4,1,4
Business services,9,6,8,9,1,4
Consumer & food,3,2,0,2,1,0
Energy & environmental,0,0,1,1,1,0
Healthcare,4,1,4,2,0,2
IT services & cybersecurity,6,8,5,12,3,2
Industrial manufacturing,10,6,7,5,4,1
Software & data,3,2,3,5,2,3
Transportation & logistics,0,1,0,0,0,0


In [15]:
# What the announcements leave out: the honest size of the gaps (a missing price is never zero)
an.disclosure_gaps(deals)

,field,missing,share_missing
0,Transaction price,115,1.000000
1,Closing date,115,1.000000
2,Ownership stake,114,0.991304
3,Target location (state),45,0.391304
4,Target description,2,0.017391
5,Stated rationale,1,0.008696
6,Deal status (completed / pending),35,0.304348


**What the evidence supports:** which companies were bought, by whom, when, and the capabilities the sponsors say they
add. **What needs further research:** prices (not disclosed), whether add-ons improved margins, growth or returns
(operating data), deals the firms did not announce. The collected announcements are not a measure of the PE market.

## 7 · Extras: similar deals, the new-deal digest, exports

In [16]:
# Who else bought something like Jones Bearing? (target description and rationale in similar words)
an.similar_deals(deals, jones["deal_id"], k=5)[["similarity", "published_date", "publisher", "buyer", "target", "target_industry"]]

,similarity,published_date,publisher,buyer,target,target_industry
0,0.132874,2025-01-07,Goldner Hawn,Matrix Adhesives Group,Roman Products,Industrial manufacturing
1,0.121108,2024-02-29,Goldner Hawn,Trystar,Macromatic; Primax Technologies,Industrial manufacturing
2,0.107487,2026-07-14,Align Capital Partners,Marco Sealing Solutions,"Rocket Seals, Inc",Value-added distribution
3,0.096262,2023-01-19,Align Capital Partners,StenTech,"Advanced Tooling Design, Inc",Industrial manufacturing
4,0.091236,2026-09-09,Align Capital Partners,MKB Company,Siltworm,Industrial manufacturing


In [17]:
# Monitoring: everything published in the last 120 days of the data
latest = pd.to_datetime(deals["published_date"]).max()
an.new_since(deals, (latest - pd.Timedelta(days=120)).date().isoformat())[["published_date", "publisher", "buyer", "target", "transaction_type"]]

,published_date,publisher,buyer,target,transaction_type
114,2026-10-13,Goldner Hawn,Goldner Hawn,International Filtration Services,Platform investment
113,2026-10-07,Align Capital Partners,Advantage Investigations,NorStar Investigations,Add-on acquisition
112,2026-09-09,Align Capital Partners,MKB Company,Siltworm,Add-on acquisition
111,2026-09-03,Align Capital Partners,Marco Sealing Solutions,Global O-Ring and Seal,Add-on acquisition
110,2026-09-02,Align Capital Partners,Align Capital Partners,Trident Solution,Platform investment
109,2026-09-01,Align Capital Partners,Blue Sea Capital,CompletePet,Exit
108,2026-08-24,Align Capital Partners,E Source,Aneden Consulting,Add-on acquisition
107,2026-07-14,Align Capital Partners,Marco Sealing Solutions,"Rocket Seals, Inc",Add-on acquisition
106,2026-07-07,Tonka Bay Equity Partners,Standard Locknut,Jones Bearing,Add-on acquisition
105,2026-07-01,Huron Capital,River Associates Investments,Horsepower Automotive Group,Exit


In [18]:
# Export: CSV / JSON / Markdown brief (saved sample outputs live in data/pe_deals/output/)
for p in ds.export("pe_deals_output"):
    print(p)

pe_deals_output/deals.csv
pe_deals_output/all_articles.csv
pe_deals_output/coverage.csv
pe_deals_output/failures.csv
pe_deals_output/deals.json
pe_deals_output/deal_brief.md


## Limitations

* Four firms' self-published announcements: selective and incomplete, not the PE market.
* A stated rationale is the sponsor's claim, not evidence that a deal improved results.
* Prices, closing dates and stakes are almost never disclosed; the fields show the gap instead of filling it.
* Pages with no publication date are excluded from the period and listed, never guessed.
* The rules fit these sites' press-release formulas; a new firm needs a source entry and a labelled check first.
* The reference labels were produced with AI assistance and still need a human spot-check (`data/pe_deals/validation/README.md`).
